## 준비. 패키지 설치와 드라이브 연결


In [1]:
%pip install -q numpy pandas faiss-cpu "sentence-transformers==5.6.0" langchain-huggingface langchain-community langchain-text-splitters pymupdf4llm langchain-hwp-hwpx-loader "transformers==4.51.3" "autoawq==0.2.9" accelerate rank_bm25


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm
from google.colab import drive
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/부트캠프/Part3/프로젝트/data"
metadata_csv = "/content/drive/MyDrive/부트캠프/Part3/프로젝트/data_list.csv"


Mounted at /content/drive


In [3]:
# 문서 순서를 고정해 다시 실행해도 청크 번호가 같은 순서로 생성되도록 합니다.
sorted_file_names = sorted(
    os.listdir(folder),
    key=lambda name: (unicodedata.normalize("NFKC", name).casefold(), name),
)
print("정렬한 파일 수:", len(sorted_file_names))

정렬한 파일 수: 99


## 1. 문서 로드 (HWP, PDF)


In [4]:
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()


df = pd.read_csv(metadata_csv, encoding="utf-8")
loaded_files = []

for name in sorted_file_names:
    full_path = os.path.join(folder, name)

    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = loader.load()[0].page_content

    loaded_files.append((name, text))

print("로드한 문서:", len(loaded_files))


/usr/local/lib/python3.13/dist-packages/pymupdf4llm/ocr/compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice.
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=4/5.
OCR on page.number=6/7.
OCR on page.number=7/8.
OCR on page.number=8/9.
OCR on page.number=9/10.
OCR on page.number=10/11.
OCR on page.number=11/12.
OCR on page.number=13/14.
OCR on page.number=14/15.
OCR on page.number=15/16.
OCR on page.number=16/17.
OCR on page.number=17/18.
OCR on page.number=18/19.
OCR on page.number=19/20.
OCR on page.number=20/21.
OCR on page.number=21/22.
OCR on page.number=22/23.
OCR on page.number=23/24.
OCR on page.number=24/25.
OCR on page.number=25/26.
OCR on page.number=26/27.
OCR on page.number=27/28.
OCR on page.number=28/29.
OCR on page.number=29/30.
OCR on page.number=30/31.
OCR on page.number=31/32.
OCR on page.number=32/33.
OCR on page.number=33/34.
OCR on page.number=34/35.
OCR on page.number=35/36.
OCR on page.number=41/42.
OCR on page.number=43/44.
OCR on page

## 2. 메타데이터와 본문을 Document로 저장

`page_content`에 본문을, `metadata`에 CSV의 문서 정보를 저장합니다.


In [5]:
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]

documents = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        continue

    row = result_df.iloc[0]
    metadata = {column: row[column] for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))

print("메타데이터와 연결한 문서:", len(documents))


메타데이터와 연결한 문서: 99


## 3. 본문 정제 및 보존검사

NFC 정규화, 손상된 선행 글머리기호, 쪽 번호 전용 줄, 점선형 목차 항목, 빈 표 줄, 연속 공백과 과도한 빈 줄을 정리합니다. 평가셋의 `evidence_quotes`에서 실제 원문에 존재하는 인용 구절을 찾아 정제 후에도 남아 있는지 확인합니다. 참조답의 요약 문장은 원문 일치 검사의 대상으로 사용하지 않습니다.


In [6]:
import json
from collections import defaultdict
from pathlib import Path


def compact_for_preservation(text):
    """Ignore only Unicode composition and whitespace when comparing source text."""
    return re.sub(r"\s+", "", unicodedata.normalize("NFC", text))


def clean_document_text(text):
    text = unicodedata.normalize("NFC", text.replace("\r\n", "\n").replace("\r", "\n"))
    kept_lines = []
    removed_lines = []
    for line in text.split("\n"):
        # 손상된 글머리기호(PUA 문자 또는 U+FFFD)가 줄 앞에 있을 때만 제거
        line = re.sub(r"^[\ue000-\uf8ff\uFFFD]+[ \t]*", "", line)
        short = line.strip()

        # 숫자만 있거나 '- 12 -' 형태인 쪽 번호 줄
        is_page_number = bool(re.fullmatch(r"(?:[-–—]\s*)?\d{1,4}(?:\s*[-–—])?", short))
        # '제목 ........ 12'와 같은 점선형 목차 항목
        is_toc_entry = bool(re.fullmatch(r".{2,200}?(?:\s*[.·…⋯]){2,}\s*\d{1,4}", short))
        is_toc_title = bool(re.fullmatch(r"목\s*차", short))
        is_empty_table_row = short.startswith("|") and all(
            not cell.strip() for cell in short.split("|")
        )
        if is_page_number or is_toc_entry or is_toc_title or is_empty_table_row:
            removed_lines.append(line)
            continue
        kept_lines.append(line)

    cleaned = "\n".join(kept_lines)
    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()
    return cleaned, removed_lines


dataset_paths = [
    Path("/content/drive/MyDrive/rag_eval_dataset.json"),
    Path("rag_eval_dataset.json"),
]
preservation_dataset_path = next((path for path in dataset_paths if path.is_file()), None)
if preservation_dataset_path is None:
    raise FileNotFoundError(
        "보존검사에 필요한 rag_eval_dataset.json을 내 드라이브 루트 또는 노트북 폴더에 두세요."
    )
with preservation_dataset_path.open(encoding="utf-8") as file:
    preservation_records = json.load(file)

# 인용부호 안의 실제 원문만 사용합니다. 줄임표가 들어간 축약 인용은 제외합니다.
protected_anchors = defaultdict(set)
for record in preservation_records:
    for evidence in record.get("evidence_quotes", []):
        for quoted in re.findall(r'[“"]([^”"]+)[”"]', evidence):
            if "…" in quoted or "..." in quoted:
                continue
            anchor = compact_for_preservation(quoted)
            if len(anchor) >= 8 and re.search(r"[가-힣A-Za-z]", anchor):
                protected_anchors[anchor].add(record["id"])

matched_anchors = set()
matched_question_ids = set()
intentional_removals = set()
lost_evidence = []

for document in documents:
    before = document.page_content
    after, removed_lines = clean_document_text(before)
    before_compact = compact_for_preservation(before)
    after_compact = compact_for_preservation(after)
    removed_compact = compact_for_preservation("\n".join(removed_lines))

    for anchor, question_ids in protected_anchors.items():
        before_count = before_compact.count(anchor)
        if before_count == 0:
            continue  # 이 PDF에서 인용 구절이 추출되지 않았으면 보존 여부도 판정 불가
        matched_anchors.add(anchor)
        matched_question_ids.update(question_ids)
        after_count = after_compact.count(anchor)
        removed_count = removed_compact.count(anchor)
        if after_count >= before_count:
            continue
        if after_count + removed_count >= before_count:
            intentional_removals.add(anchor)
            continue
        lost_evidence.append({
            "file": document.metadata.get("파일명", "파일명 확인 불가"),
            "question_ids": sorted(question_ids),
            "quote": anchor,
            "before_count": before_count,
            "after_count": after_count,
            "intentionally_removed_count": removed_count,
        })

    document.page_content = after

print(f"정제한 문서: {len(documents)}개")
print(
    f"원문 인용 구절 검사: {len(matched_anchors)}/{len(protected_anchors)}개 확인, "
    f"인용 문자열이 발견된 평가문항 {len(matched_question_ids)}/{len(preservation_records)}개"
)
if intentional_removals:
    print(f"목차·쪽번호·빈 표 줄에서만 발견되어 제외한 인용 구절: {len(intentional_removals)}개")
if not matched_anchors:
    print("보존검사 판정 불가: 평가셋 인용 구절이 로드된 원문에서 확인되지 않았습니다.")
if lost_evidence:
    print("정제 후 사라진 인용 구절 예시:", lost_evidence[:5])
    raise ValueError(f"보존검사 실패: 원문 인용 구절 {len(lost_evidence)}건이 사라졌습니다.")


FileNotFoundError: 보존검사에 필요한 rag_eval_dataset.json을 내 드라이브 루트 또는 노트북 폴더에 두세요.

## 4. 청킹


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))


## 5. 임베딩 후 FAISS DB에 저장

`embedding_model_name`을 바꾸면 임베딩 모델을 조정할 수 있습니다.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model_name = "Alibaba-NLP/gte-multilingual-base"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
)

vector_store = FAISS.from_documents(chunks, embeddings)
print("저장한 청크:", vector_store.index.ntotal)


## 6. 평가 질문 불러오기

`rag_eval_dataset.json`을 Google Drive의 `내 드라이브` 루트에 올려 두고 실행합니다. 로컬에서 실행할 때는 노트북과 같은 폴더에 둡니다. 전체 180문항을 사용하며, 필요한 경우 아래 셀의 `EVAL_PROJECTS` 또는 `EVAL_LIMIT`으로 범위를 줄일 수 있습니다.


In [ ]:
import json
from collections import Counter
from pathlib import Path

eval_dataset_candidates = [
    Path("/content/drive/MyDrive/rag_eval_dataset.json"),
    Path("rag_eval_dataset.json"),
]
eval_dataset_path = next((path for path in eval_dataset_candidates if path.is_file()), None)
if eval_dataset_path is None:
    raise FileNotFoundError(
        "rag_eval_dataset.json을 Google Drive 내 드라이브 루트 또는 노트북 폴더에 두세요."
    )

with eval_dataset_path.open(encoding="utf-8") as file:
    all_eval_records = json.load(file)

EVAL_PROJECTS = None  # 예: {"수문자료정보관리시스템(HDIMS) 재구축사업(3단계)"}
EVAL_LIMIT = None     # 빠른 시범 실행 시 예: 12. None이면 전부 사용합니다.

selected_eval_records = [
    record for record in all_eval_records
    if EVAL_PROJECTS is None or record["project"] in EVAL_PROJECTS
]
if EVAL_LIMIT is not None:
    selected_eval_records = selected_eval_records[:EVAL_LIMIT]
if not selected_eval_records:
    raise ValueError("선택된 평가 문항이 없습니다. 사업명 필터와 입력 파일을 확인하세요.")

evaluation_data = []
for record in selected_eval_records:
    if not record.get("id") or not record.get("question") or not record.get("reference"):
        raise ValueError(f"필수 평가 필드가 비어 있습니다: {record.get('id')}")
    evaluation_data.append({
        "id": record["id"],
        "query": record["question"],
        "reference": record["reference"],
        "project": record["project"],
        "question_type": record["question_type"],
    })

if len({item["id"] for item in evaluation_data}) != len(evaluation_data):
    raise ValueError("평가 문항 ID가 중복됩니다.")
if len({item["query"] for item in evaluation_data}) != len(evaluation_data):
    raise ValueError("평가 질문이 중복됩니다.")

question = evaluation_data  # 기존 검색·답변 생성 반복문에서 사용
print(f"평가셋: {eval_dataset_path} / {len(evaluation_data)}문항")
print("유형별:", dict(Counter(item["question_type"] for item in evaluation_data)))
print("사업별:", dict(Counter(item["project"] for item in evaluation_data)))


## 7. 하이브리드 검색 (Dense + BM25, RRF)


In [ ]:
from langchain_community.retrievers import BM25Retriever

dense_retriever = vector_store.as_retriever(search_kwargs={"k": 8})
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 8


def rrf_search(query, k=60):
    dense_results = dense_retriever.invoke(query)
    sparse_results = bm25_retriever.invoke(query)
    scores = {}

    for results in (dense_results, sparse_results):
        for rank, doc in enumerate(results, start=1):
            doc_id = doc.metadata["chunk_id"]
            if doc_id not in scores:
                scores[doc_id] = {"doc": doc, "score": 0}
            scores[doc_id]["score"] += 1 / (k + rank)

    ranked = sorted(scores.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ranked]


## 8. HyDE로 검색 문장 만들기


In [ ]:
hyde_search_rules = '''너는 RFP에서 관련 문단을 찾기 위한 가상 안내문 작성자야.
실제 답변이 아니라 검색에 쓸 안내문 1문장만 작성해.
- 질문의 핵심 주제어를 그대로 포함해.
- 질문과 관계없는 다른 사업이나 항목으로 이동하지 마.
- 확인되지 않은 금액, 날짜, 법률, 자격 조건을 단정하지 마.
- 제목, 목록, 따옴표, 추가 설명을 쓰지 마.

예시:
질문: 이 사업의 입찰 참여 자격은 무엇인가요?
가상 안내문: 이 사업의 입찰 참여 자격에 관한 조건이 안내되어 있다.'''

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace


language_model_name = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ"
model_revision = "f2699f35d5b4ba511ab93826c22bd07370296b5c"


model = AutoModelForCausalLM.from_pretrained(
    language_model_name, torch_dtype=torch.float16,
    trust_remote_code=True, device_map="auto", revision=model_revision
)
tokenizer = AutoTokenizer.from_pretrained(language_model_name, revision=model_revision)


llm_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    repetition_penalty=1.2,
    return_full_text=False,
    max_new_tokens=220,
)


llm = HuggingFacePipeline(pipeline=llm_pipeline)
chat_model = ChatHuggingFace(llm=llm)

method = "basic"  # "basic" 또는 "hyde"


def make_search_text(query, method):
    if method == "basic":
        return query
    if method == "hyde":
        hyde_prompt = f"{hyde_search_rules}\n\n질문: {query}\n가상 안내문:"
        return chat_model.invoke(hyde_prompt).content.strip()
    raise ValueError(f"지원하지 않는 검색 방식: {method}")


## 9. 재순위화


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True,
    device="cpu"
)

# 확인 셀과 답변 생성 셀이 같은 검색·재순위화 결과를 공유합니다.
retrieval_cache = {}


def retrieve_and_rerank(query, method):
    cache_key = (method, query)
    if cache_key not in retrieval_cache:
        search_text = make_search_text(query, method)
        found = rrf_search(search_text)
        ranking = reranker.rank(
            query,  # 재순위화에는 원래 질문 사용
            [doc.page_content for doc in found],
            top_k=4,
        )
        reranked_docs = [found[row["corpus_id"]] for row in ranking]
        retrieval_cache[cache_key] = (search_text, found, reranked_docs)
    return retrieval_cache[cache_key]


for i, q in enumerate(question, start=1):
    query = q["query"]
    search_text, found, reranked_docs = retrieve_and_rerank(query, method)
    print(f"\n{i}번째 질문: {query}")
    print("검색 문장:", search_text)
    print("리랭크 전:", [doc.metadata["chunk_id"] for doc in found])
    print("리랭크 후:", [doc.metadata["chunk_id"] for doc in reranked_docs])


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

## 10. 답변 생성


In [ ]:
# 위 셀의 검색·재순위화 결과로 최종 답변을 한 번만 생성합니다.
answer_rules = '''
너는 RFP 문서의 사실 확인 담당자다. 아래 <문서>에 적힌 내용만으로 <질문>에 답한다.

규칙:
1. 질문이 여러 항목을 묻는다면 항목별로 근거를 따로 찾는다.
2. 질문의 사업명과 다른 사업의 정보는 사용하지 않는다.
3. 각 항목에 대해 문서에서 직접 인용할 수 있는 문구를 먼저 제시한다.
4. 직접 인용할 근거가 없는 항목은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 쓴다.
5. 금액, 기간, 날짜, 비율, 인력, 제출 조건은 문서의 표현을 그대로 옮긴다. 계산하거나 일반적인 관행으로 빈칸을 채우지 않는다.
6. 답변의 각 사실 바로 뒤에 근거 문서 번호 [1]처럼 붙인다.
7. HyDE 검색 문장과 모델의 사전 지식은 근거로 사용하지 않는다.

출력 형식:
근거:
- 항목명: [문서 번호] "문서에서 그대로 가져온 짧은 문구"
- 근거가 없는 항목명: 확인 불가

답변:
근거가 확인된 내용만 간결하게 답한다. 근거가 없는 항목은 확인할 수 없다고 밝힌다.

<문서>
{context}
</문서>

<질문>
{query}
</질문>
'''

answers = []
for i, item in enumerate(evaluation_data, start=1):
    query = item["query"]
    search_text, found, reranked_docs = retrieve_and_rerank(query, method)
    context = "\n\n".join(
        f"[{j}] 메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for j, doc in enumerate(reranked_docs, start=1)
    )
    answer = chat_model.invoke(
        answer_rules.format(context=context, query=query)
    ).content.strip()

    answers.append({
        "query": query,
        "search_text": search_text,
        "answer": answer,
        "contexts": [doc.page_content for doc in reranked_docs],
        "chunk_ids": [doc.metadata["chunk_id"] for doc in reranked_docs],
    })
    print(f"{i}번째 질문: {query}")
    print("사용 청크:", answers[-1]["chunk_ids"])
    print("답변:", answer, "\n")


## 11. RAGAS 평가

위에서 만든 `answers`를 다시 생성하지 않고 평가합니다. `LLMContextRecall`은 참조답의 사실이 검색 문맥에 얼마나 포함되는지, `Faithfulness`는 답변이 검색 문맥에 의해 뒷받침되는지, `FactualCorrectness`는 답변이 참조답과 얼마나 일치하는지 채점합니다. 평가에는 위에서 사용한 로컬 `chat_model`을 사용합니다. 180문항 전체 평가는 시간이 오래 걸릴 수 있습니다.


In [ ]:
%pip install -q "ragas==0.3.9"


In [ ]:
# 이미 만든 검색 결과와 답변만 평가합니다. 답변 생성 셀을 먼저 실행하세요.
import pandas as pd
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import FactualCorrectness, Faithfulness, LLMContextRecall
from ragas.run_config import RunConfig

if "answers" not in globals() or len(answers) != len(evaluation_data):
    raise ValueError("평가 질문을 불러온 뒤 답변 생성 셀을 실행해 주세요.")

answer_by_query = {answer["query"]: answer for answer in answers}
if len(answer_by_query) != len(answers):
    raise ValueError("생성 답변에 중복 질문이 있습니다.")

ragas_samples = []
for item in evaluation_data:
    answer_record = answer_by_query.get(item["query"])
    if answer_record is None:
        raise ValueError(f"생성 답변이 없습니다: {item['id']}")
    full_answer = answer_record["answer"].strip()
    # 답변 프롬프트의 '근거:' 인용 목록은 출력에서 제외하고 최종 답변만 채점합니다.
    response = full_answer.split("답변:", 1)[-1].strip()
    if not response or not answer_record["contexts"]:
        raise ValueError(f"답변 또는 검색 문맥이 비어 있습니다: {item['id']}")
    ragas_samples.append({
        "user_input": item["query"],
        "retrieved_contexts": [str(context) for context in answer_record["contexts"]],
        "response": response,
        "reference": item["reference"],
    })

ragas_dataset = EvaluationDataset.from_list(ragas_samples)
ragas_evaluator_llm = LangchainLLMWrapper(chat_model)
ragas_result = evaluate(
    dataset=ragas_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=ragas_evaluator_llm,
    run_config=RunConfig(max_workers=1, max_retries=1, timeout=180),
    raise_exceptions=True,
    show_progress=True,
)

evaluation_df = ragas_result.to_pandas()
evaluation_df.insert(0, "id", [item["id"] for item in evaluation_data])
evaluation_df.insert(1, "project", [item["project"] for item in evaluation_data])
evaluation_df.insert(2, "question_type", [item["question_type"] for item in evaluation_data])
evaluation_df["selected_chunk_ids"] = [
    json.dumps(answer_by_query[item["query"]]["chunk_ids"], ensure_ascii=False)
    for item in evaluation_data
]

results_path = eval_dataset_path.with_name("ragas_eval_results.csv")
evaluation_df.to_csv(results_path, index=False, encoding="utf-8-sig")
metric_columns = [
    column for column in evaluation_df.columns
    if column in {"context_recall", "llm_context_recall", "faithfulness", "factual_correctness"}
]
display(evaluation_df[["id", "project", "question_type", *metric_columns]])
print("평균 점수:")
print(evaluation_df[metric_columns].mean(numeric_only=True).to_string())
print(f"전체 결과 저장: {results_path}")
